# Pair Trade Backtest Template

A simple template for signal generation and PnL tracking on a beta-hedged return spread.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from db.connection import get_engine
from stores.market import PriceStore

In [ ]:
DATA_BACKEND = "local"
APP_ENV = "uat"
PAIR = ("HYG", "JNK")
START_DATE = "2023-01-01"
hedge_window = 60
z_window = 20
entry_z = 2.0
exit_z = 0.5

In [ ]:
engine = get_engine(data_backend=DATA_BACKEND, app_env=APP_ENV)
price_store = PriceStore(engine)
histories = price_store.get_multi_ticker_price_history(list(PAIR), start_date=START_DATE)
prices = pd.DataFrame({ticker: frame["adj_close"] for ticker, frame in histories.items()}).dropna()
returns = np.log(prices).diff().dropna()
returns.columns = [f"{PAIR[0]}_ret", f"{PAIR[1]}_ret"]
returns.tail()

In [ ]:
cov_ab = returns[f"{PAIR[0]}_ret"].rolling(hedge_window).cov(returns[f"{PAIR[1]}_ret"])
var_b = returns[f"{PAIR[1]}_ret"].rolling(hedge_window).var()
beta = cov_ab / var_b

spread = returns[f"{PAIR[0]}_ret"] - beta * returns[f"{PAIR[1]}_ret"]
spread_mean = spread.rolling(z_window).mean()
spread_std = spread.rolling(z_window).std(ddof=0)
zscore = (spread - spread_mean) / spread_std

df = pd.DataFrame({
    "beta": beta,
    "spread": spread,
    "zscore": zscore,
}).dropna()
df.tail()

In [ ]:
signal = pd.Series(0, index=df.index, dtype=float)
signal[df["zscore"] > entry_z] = -1.0
signal[df["zscore"] < -entry_z] = 1.0
signal = signal.replace(0, np.nan).ffill().fillna(0.0)
signal[df["zscore"].abs() < exit_z] = 0.0
signal = signal.ffill().fillna(0.0)

df["position"] = signal
df["strategy_return"] = df["position"].shift(1).fillna(0.0) * df["spread"]
df["cum_return"] = df["strategy_return"].cumsum()

df.tail()

In [ ]:
print("Final cumulative log return:", round(df["cum_return"].iloc[-1], 4))
print("Latest z-score:", round(df["zscore"].iloc[-1], 4))
print("Latest beta:", round(df["beta"].iloc[-1], 4))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

df["zscore"].plot(ax=axes[0], title="Z-Score")
axes[0].axhline(entry_z, color="red", linestyle="--")
axes[0].axhline(-entry_z, color="green", linestyle="--")
axes[0].axhline(0, color="black", linestyle="--", alpha=0.7)

df["position"].plot(ax=axes[1], title="Position")

df["cum_return"].plot(ax=axes[2], title="Cumulative Strategy Return")
axes[2].axhline(0, color="black", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()